# LCO-HBI Analysis for an optimized PyPSA network

This notebook loads an optimized PyPSA network (.nc), inspects network structure, and computes a detailed, traceable decomposition of the Levelized Cost of HBI (LCO-HBI).

Usage: set the `network_file` parameter in Cell 2 and run all cells. The notebook attempts to reuse helper functions from `workflow/scripts/calculate_lcox.py` for validation where available.

In [9]:
# Imports and helper functions
import sys
import pathlib
import seaborn as sns

sns.set_theme(style="whitegrid")

# Add repository root to path so we can import workflow scripts
repo_root = pathlib.Path(
    ".."
).resolve()  # notebook lives in workflow/notebooks; adjust when running from repo root
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Parameters - edit these before running
network_file = str(
    repo_root.parent
    / "resources"
    / "lco-hbi"
    / "cost_year~2050"
    / "Europe_reserved"
    / "network_10.nc"
)  # path relative to repo root
product = "hbi"  # product to evaluate (hbi)
region = None  # optional: restrict to a region (string)

In [10]:
try:
    import pypsa
except Exception as e:
    raise RuntimeError(
        "PyPSA import failed. Make sure the environment has pypsa installed."
    ) from e

# Optional: import repo LCoX utilities if available
calc_lcox = None
try:
    from workflow.scripts import calculate_lcox as calculate_lcox_module

    calc_lcox = calculate_lcox_module
    print("Imported workflow.scripts.calculate_lcox")
except Exception:
    try:
        # fallback to relative import if executed from repository root
        import workflow.scripts.calculate_lcox as calculate_lcox_module

        calc_lcox = calculate_lcox_module
        print("Imported workflow.scripts.calculate_lcox")
    except Exception:
        print(
            "calculate_lcox module not available; notebook will still run but some validations will be skipped"
        )


# CRF helper
def crf(r, n):
    if r <= 0:
        return 1.0 / n
    return (r * (1 + r) ** n) / (((1 + r) ** n) - 1)


def annualize(capex, r, n):
    """Annualize capital cost (capex) using discount rate r and lifetime n (years)."""
    return capex * crf(r, n)

calculate_lcox module not available; notebook will still run but some validations will be skipped


In [19]:
# Load network
import pypsa

print("Loading network:", network_file)
n = pypsa.Network(network_file)
print("Network loaded.")

# Basic checks
print("Snapshots:", len(n.snapshots) if n.snapshots is not None else "None")
obj = None
try:
    obj = getattr(n, "objective", None) or getattr(n, "objective_value", None)
    print("Objective / objective_value:", obj)
except Exception:
    print(
        "Objective not found on network object (this may be a network saved before extract)."
    )

# Quick network summary
print("\nComponent counts:")
for c in ["buses", "generators", "links", "lines", "storages"]:
    comp = getattr(n, c, None)
    if comp is not None:
        print(f" - {c}:", len(comp))

# show top generator capacities
if hasattr(n, "generators") and len(n.generators) > 0:
    gcap = (
        n.generators["p_nom_opt"]
        .fillna(n.generators.get("p_nom", 0))
        .sort_values(ascending=False)
        .head(10)
    )
    print("\nTop generators by optimized capacity (head 10):")
    display(gcap.head(10))

# show buses of interest
if hasattr(n, "buses") and "region" in n.buses.columns:
    print("\nBus regions sample:")
    display(n.buses["region"].value_counts().head(10))

INFO:pypsa.network.io:New version 1.2.1 available! (Current: 1.1.2)


Loading network: C:\Users\JanLeopoldTautorus\Repos\shift\resources\lco-hbi\cost_year~2050\Europe_reserved\network_10.nc


INFO:pypsa.network.io:Imported network 'LCOX-Europe-hbi-10.0' has buses, carriers, generators, links, loads, stores, sub_networks


Network loaded.
Snapshots: 8760
Objective / objective_value: 5463430058.465736

Component counts:
 - buses: 7
 - generators: 32
 - links: 2
 - lines: 0

Top generators by optimized capacity (head 10):


name
iron_ore                           1.000000e+10
grid_electricity_input             1.000000e+10
battery_input                      1.000000e+10
renewable_Europe_windonshore_30    9.413415e+03
renewable_Europe_windonshore_27    7.260414e+03
renewable_Europe_pvplant_17        5.731441e+03
renewable_Europe_pvplant_16        3.627389e-09
renewable_Europe_pvplant_15        1.018952e-09
renewable_Europe_pvplant_14        7.960459e-10
renewable_Europe_pvplant_13        5.860929e-10
Name: p_nom_opt, dtype: float64

In [24]:
n.buses

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network
name,,,,,,,,,,,,,
renewable_electricity,1.0,,0.0,0.0,renewable_electricity,MW,,1.0,0.0,inf,Slack,renewable_Europe_windoffshore_0,0
hydrogen,1.0,,0.0,0.0,hydrogen,MW,,1.0,0.0,inf,Slack,,1
battery,1.0,,0.0,0.0,battery_elec,MWh,,1.0,0.0,inf,Slack,battery_input,2
iron_ore,1.0,,0.0,0.0,iron_ore,t/h,,1.0,0.0,inf,Slack,iron_ore,3
hbi,1.0,,0.0,0.0,hbi,t/h,,1.0,0.0,inf,Slack,,4
steel,1.0,,0.0,0.0,steel,t/h,,1.0,0.0,inf,Slack,,5
grid_electricity,1.0,,0.0,0.0,grid_electricity,t/h,,1.0,0.0,inf,Slack,grid_electricity_input,6


In [25]:
n.carriers

,co2_emissions,color,nice_name,max_growth,max_relative_growth,description
name,,,,,,
renewable_electricity,0.0,#17becf,,inf,0.0,Islanded renewable electricity
renewable_pv,0.0,#1f77b4,,inf,0.0,Photovoltaic (utility)
renewable_wind_onshore,0.0,#2ca02c,,inf,0.0,Onshore wind
renewable_wind_offshore,0.0,#ff7f0e,,inf,0.0,Offshore wind
hydrogen,0.0,#7f7f7f,,inf,0.0,Hydrogen gas
battery_elec,0.0,#ff7f0e,,inf,0.0,Battery (electrical energy)
iron_ore,0.0,#bcbd22,,inf,0.0,Iron ore (mass)
hbi,0.0,#e377c2,,inf,0.0,Hot Briquetted Iron (mass)
steel,0.0,#d62728,,inf,0.0,Steel (mass)


In [26]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,p_nom_opt,bus3,efficiency3,bus2,efficiency2
name,,,,,,,,,,,,,,,,,,,,,
electrolysis,renewable_electricity,hydrogen,,electrolysis,0.724638,True,0,40.0,0.0,0.0,...,0,NaN,NaN,NaN,NaN,7445.481727,,1.00,,1.0
dri,iron_ore,hbi,,direct_reduction_furnace,0.628931,True,0,40.0,0.0,0.0,...,0,NaN,NaN,NaN,NaN,1810.482160,renewable_electricity,-1.03,hydrogen,-2.1


In [28]:
n.buses_t.marginal_price

name,renewable_electricity,hydrogen,battery,iron_ore,hbi,steel,grid_electricity
snapshot,,,,,,,
2050-01-01 00:00:00,1.647197e-13,5.975359e-13,-10365.300193,-5.684342e-14,557.055614,0.0,0.0
2050-01-01 01:00:00,1.541077e-13,6.002193e-13,-10364.927651,-5.684342e-14,557.055614,0.0,0.0
2050-01-01 02:00:00,1.500782e-13,6.029611e-13,-10364.629458,-1.136868e-13,557.055614,0.0,0.0
2050-01-01 03:00:00,1.442742e-13,6.057625e-13,-10364.310784,-1.136868e-13,557.055614,0.0,0.0
2050-01-01 04:00:00,1.493886e-13,6.086263e-13,-10365.300193,-5.684342e-14,557.055614,0.0,0.0
...,...,...,...,...,...,...,...
2050-12-31 19:00:00,2.501217e-13,5.850567e-13,-10364.310784,-1.136868e-13,557.055614,0.0,0.0
2050-12-31 20:00:00,2.512324e-13,5.874974e-13,-10365.300193,-5.684342e-14,557.055614,0.0,0.0
2050-12-31 21:00:00,2.571686e-13,5.899859e-13,-10364.927651,-1.136868e-13,557.055614,0.0,0.0


In [29]:
n.generators_t.p

name,iron_ore,grid_electricity_input,battery_input,renewable_Europe_windoffshore_0,renewable_Europe_windoffshore_25,renewable_Europe_windoffshore_26,renewable_Europe_windoffshore_27,renewable_Europe_windoffshore_28,renewable_Europe_windoffshore_29,renewable_Europe_windoffshore_30,...,renewable_Europe_windonshore_21,renewable_Europe_windonshore_22,renewable_Europe_windonshore_23,renewable_Europe_windonshore_24,renewable_Europe_windonshore_25,renewable_Europe_windonshore_26,renewable_Europe_windonshore_27,renewable_Europe_windonshore_28,renewable_Europe_windonshore_29,renewable_Europe_windonshore_30
snapshot,,,,,,,,,,,,,,,,,,,,,
2050-01-01 00:00:00,1810.48216,0.0,1.634661e-14,0.0,3.634305e-13,1.825851e-12,1.125964e-12,1.265568e-11,1.401227e-11,1.225096e-11,...,1.640382e-13,5.759051e-13,5.264248e-13,8.678800e-13,3.649729e-12,1.981407e-11,3273.971847,2.698786e-11,1.933776e-11,5719.128703
2050-01-01 01:00:00,1810.48216,0.0,1.498259e-14,0.0,2.413563e-13,1.640004e-12,1.025071e-12,1.148651e-11,1.376569e-11,1.188466e-11,...,1.551878e-13,5.457161e-13,5.051965e-13,8.734515e-13,3.600607e-12,1.986972e-11,3200.283663,2.708063e-11,1.924072e-11,5827.209737
2050-01-01 02:00:00,1810.48216,0.0,1.547255e-14,0.0,1.965869e-13,1.514972e-12,9.309111e-13,9.517512e-12,1.365411e-11,1.176995e-11,...,1.437529e-13,4.954655e-13,4.648807e-13,8.500322e-13,3.575491e-12,1.986025e-11,3130.998637,2.649550e-11,1.906257e-11,5908.282947
2050-01-01 03:00:00,1810.48216,0.0,1.422767e-14,0.0,2.223022e-13,1.542584e-12,8.963757e-13,8.376632e-12,1.361448e-11,1.170980e-11,...,1.400023e-13,4.784219e-13,4.530637e-13,8.490766e-13,3.605771e-12,1.972301e-11,3050.947854,2.590942e-11,1.887819e-11,6003.723784
2050-01-01 04:00:00,1810.48216,0.0,1.634661e-14,0.0,1.678554e-13,1.509112e-12,8.213856e-13,7.440544e-12,1.352749e-11,1.152004e-11,...,1.366645e-13,4.744644e-13,4.571818e-13,8.583457e-13,3.635376e-12,1.969108e-11,3004.484606,2.547251e-11,1.874917e-11,6041.265360
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2050-12-31 19:00:00,1810.48216,0.0,1.422767e-14,0.0,7.055640e-13,2.878465e-12,3.400270e-12,3.060704e-11,1.432128e-11,1.551852e-11,...,5.157141e-14,2.406933e-13,3.380474e-13,7.841299e-13,2.919722e-12,1.436480e-11,2502.605814,2.609376e-11,2.062582e-11,6016.807461
2050-12-31 20:00:00,1810.48216,0.0,1.634661e-14,0.0,6.579918e-13,2.704858e-12,3.117140e-12,2.975738e-11,1.422818e-11,1.531806e-11,...,5.360731e-14,2.546093e-13,3.484719e-13,7.877509e-13,2.887056e-12,1.425297e-11,2471.891442,2.621920e-11,2.059685e-11,6065.930453
2050-12-31 21:00:00,1810.48216,0.0,1.498259e-14,0.0,7.537291e-13,2.542414e-12,3.014929e-12,3.015236e-11,1.418334e-11,1.524312e-11,...,5.681148e-14,2.648511e-13,3.541481e-13,8.010552e-13,2.880279e-12,1.400078e-11,2435.718675,2.663790e-11,2.055178e-11,6097.924484


In [36]:
EPSILON = 1e-9
opt_gen = n.generators.p_nom_opt
opt_gen = opt_gen[opt_gen > EPSILON]
opt_gen

name
iron_ore                           1.000000e+10
grid_electricity_input             1.000000e+10
battery_input                      1.000000e+10
renewable_Europe_pvplant_15        1.018952e-09
renewable_Europe_pvplant_16        3.627389e-09
renewable_Europe_pvplant_17        5.731441e+03
renewable_Europe_windonshore_27    7.260414e+03
renewable_Europe_windonshore_30    9.413415e+03
Name: p_nom_opt, dtype: float64